In [22]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
df = pd.read_csv("/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [13]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [14]:
df.shape


(50000, 2)

In [15]:
df = df.iloc[:10000]
df.shape

(10000, 2)

In [16]:
df['sentiment'].value_counts()

,count
sentiment,
positive,5028
negative,4972


In [18]:
df.isnull().sum()

,0
review,0
sentiment,0


In [19]:
df.duplicated().sum()

np.int64(17)

In [20]:
df.drop_duplicates(inplace=True)

In [23]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

df['review'] = df['review'].apply(remove_tags)

In [24]:
df['review'] = df['review'].apply(lambda x: x.lower())

In [25]:
sw_list = stopwords.words('english')

df['review'] = df['review'].apply(
    lambda x: " ".join([word for word in x.split() if word not in sw_list])
)

In [26]:
X = df['review']
y = df['sentiment']

In [27]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

In [29]:
cv = CountVectorizer()

X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

In [30]:
gnb = GaussianNB()
gnb.fit(X_train_bow, y_train)

y_pred = gnb.predict(X_test_bow)

accuracy_score(y_test, y_pred)

0.6324486730095142

In [31]:
confusion_matrix(y_test, y_pred)

array([[717, 235],
       [499, 546]])

In [32]:
rf = RandomForestClassifier()

rf.fit(X_train_bow, y_train)

y_pred = rf.predict(X_test_bow)

accuracy_score(y_test, y_pred)

0.8447671507260891

In [33]:
cv = CountVectorizer(max_features=3000)

X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow, y_train)
y_pred = rf.predict(X_test_bow)

accuracy_score(y_test, y_pred)

0.8367551326990486

In [34]:
cv = CountVectorizer(ngram_range=(1,2), max_features=5000)

X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow, y_train)

y_pred = rf.predict(X_test_bow)

accuracy_score(y_test, y_pred)

0.8432648973460191

In [35]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test)

In [36]:
rf = RandomForestClassifier()

rf.fit(X_train_tfidf, y_train)

y_pred = rf.predict(X_test_tfidf)

accuracy_score(y_test, y_pred)

0.8482724086129194